### **Stage F8 — Cancellation & Refund**
**Goal:** Handle booking cancellations and reviews

**Features:**
1. **Cancel Button** (on booking detail page)
   - Shows policy warning (24h rule)
   - Confirmation modal
   - Success/error feedback

2. **Refund Button** (motard-only view)
   - Motard can refund any paid booking
   - Confirmation modal
   - Updates booking and payout status

3. **Reviews**
   - After tour date, show "Leave Review" button
   - Rating stars (1-5)
   - Optional comment text area
   - Submit button

**Backend Endpoints Used:**
- `POST /api/bookings/{id}/cancel`
- `POST /api/bookings/{id}/refund`
- `POST /api/motards/{userId}/reviews`

**DoD:**
- Cancellation works with proper validation
- Refund updates payout status
- Reviews display on motard profile after submission

---


TODO:
A4: Paid Booking <24h Scenario ⚠️ (Need to modify tour date)
Option 1: Quick Backend Test

Modify DbSeeder to create a tour for January 2, 2026 (tomorrow)
Follow same flow as A3
✅ VERIFY: Cancel button is hidden OR disabled
✅ VERIFY: Warning text displays: "Less than 24h before tour — please contact the motard"

TEST C: Backend Authority & Error Handling
C1: Backend Rejects Invalid Cancel ✅
Login as participant with a paid booking <24h from tour
Use browser DevTools Console to force API call:
✅ VERIFY: Response is 400 Bad Request
✅ VERIFY: Error message explains 24h rule
✅ VERIFY: UI snackbar shows the error properly
C2: Backend Rejects Wrong User Refund ✅
Login as participant
Try to refund a booking (should fail - participant can't refund)
✅ VERIFY: 403 Forbidden or appropriate error
✅ VERIFY: Snackbar shows "You do not have permission..."

## ✅ **Bugs Fixed Summary**

### **Bug 1: Capacity Release** ✅
- **Problem**: Cancel released seats even for Pending/Approved (which never reserved seats)
- **Fix**: Only release seats if booking was `Paid` status
- **Result**: Capacity correctly managed (3/3 stays 3/3 for non-paid cancellations)

### **Bug 2: Dashboard Sections** ✅
- **Problem**: No "Cancelled Bookings" section in participant dashboard
- **Fix**: Added `cancelledBookings` section showing Cancelled/Refunded bookings
- **Result**: Better transparency for participants

### **Bug 3: Motard Sees Pay Button** ✅
- **Problem**: Motards saw "Pay Now" button on Approved bookings
- **Fix**: Added participant check to `canPay()` method
- **Result**: Only participants see payment button

---

## 🧪 **Test D: Date Format & Timezone Verification**

### **D1: Verify ISO 8601 Date Format**

**Method 1: Browser DevTools Network Tab**
1. Open booking detail page in browser
2. Press F12 → Network tab
3. Clear network log (trash icon)
4. Refresh the page (F5)
5. Look for request: `GET localhost:5256/api/bookings/{guid}`
6. Click on the request → Preview or Response tab
7. **✅ VERIFY**: `tourStartAt` field shows format like:
   - `"2026-01-04T10:00:00Z"` (UTC with Z)
   - OR `"2026-01-04T10:00:00+00:00"` (UTC with offset)

**Method 2: Console Direct API Call**
```javascript
// In browser console (F12 → Console tab):
fetch('http://localhost:5256/api/bookings', {
  credentials: 'include'
})
.then(r => r.json())
.then(data => {
  console.log('First booking tourStartAt:', data[0]?.tourStartAt);
  console.log('Format valid:', /^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}/.test(data[0]?.tourStartAt));
  return data;
});
```

**Expected Output**:
```
First booking tourStartAt: 2026-01-04T10:00:00Z
Format valid: true
```

---

### **D2: Verify Timezone Consistency**

**Test Setup**: 
- Backend uses UTC (`DateTimeKind.Utc`)
- Frontend parses ISO string and displays in French locale

**Verification Steps**:

**Step 1: Check Backend Time Format**
```javascript
// Browser console - get raw API response
fetch('http://localhost:5256/api/bookings', {credentials: 'include'})
.then(r => r.json())
.then(data => {
  const tourStart = data[0]?.tourStartAt;
  console.log('📡 Backend UTC time:', tourStart);
  
  // Parse as Date object
  const date = new Date(tourStart);
  console.log('🌍 Browser parsed (local):', date.toString());
  console.log('⏰ UTC values:', {
    year: date.getUTCFullYear(),
    month: date.getUTCMonth() + 1,
    day: date.getUTCDate(),
    hour: date.getUTCHours(),
    minute: date.getUTCMinutes()
  });
});
```

**Expected**:
```
📡 Backend UTC time: 2026-01-04T10:00:00Z
🌍 Browser parsed (local): Sat Jan 04 2026 11:00:00 GMT+0100 (Paris time)
⏰ UTC values: { year: 2026, month: 1, day: 4, hour: 10, minute: 0 }
```

**Step 2: Verify 24h Calculation**
```javascript
// Check if 24h logic works correctly
const tourStartAt = "2026-01-04T10:00:00Z";
const tourStartTime = new Date(tourStartAt).getTime();
const now = new Date().getTime();
const hoursUntilTour = (tourStartTime - now) / (1000 * 60 * 60);

console.log('Current time UTC:', new Date().toISOString());
console.log('Tour start UTC:', tourStartAt);
console.log('Hours until tour:', Math.round(hoursUntilTour * 10) / 10);
console.log('Is within 24h?', hoursUntilTour < 24);
```

**Expected** (on Jan 1, 2026):
```
Current time UTC: 2026-01-01T11:41:00.000Z
Tour start UTC: 2026-01-04T10:00:00Z
Hours until tour: 70.3
Is within 24h? false
```

**Step 3: Verify UI Display Format**
1. Look at booking detail page
2. **✅ VERIFY**: Date displays as "sam. 4 janv. 2026, 10:00" (French format)
3. **✅ VERIFY**: Time shown respects `Europe/Paris` timezone (10:00 UTC = 11:00 Paris in winter)

---

### **Quick Pass/Fail Checklist**

✅ **PASS if**:
- Backend returns ISO 8601 with timezone (`Z` or `+00:00`)
- JavaScript `new Date()` parses correctly
- 24h calculation uses milliseconds since epoch (timezone-independent)
- UI displays in French locale
- Cancel button shows/hides based on correct 24h calculation

❌ **FAIL if**:
- Date string has no timezone info
- 24h calculation gives wrong result
- UI says ">24h" but backend rejects cancel
- Times display incorrectly in UI

---